# Pocket-Agent Hackathon Pipeline (Colab Extension)

This notebook connects to your Colab GPU instance and automatically pulls your code from GitHub so that all the `make` commands work seamlessly.

In [1]:
# Step 1: Pull your code from GitHub into the Colab server
!git clone https://github.com/MAbdullahWaqar/pocket-agent-1.git pocket-agent-cloud
%cd pocket-agent-cloud
print("✅ Code successfully downloaded from GitHub!")

fatal: destination path 'pocket-agent-cloud' already exists and is not an empty directory.
/content/pocket-agent-cloud
✅ Code successfully downloaded from GitHub!


In [2]:
# Step 2: Install requirements
!make install

pip install -r requirements.txt


In [3]:
# Step 3: Generate the synthetic dataset
!make data

python data/generate_data.py
Generating data...
Generated 1500 examples. Saved to /content/pocket-agent-cloud/data/train.jsonl.


In [4]:
!git pull


remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (4/4), done.
Unpacking objects: 100% (11/11), 20.25 KiB | 3.38 MiB/s, done.
remote: Total 11 (delta 5), reused 11 (delta 5), pack-reused 0 (from 0)
From https://github.com/MAbdullahWaqar/pocket-agent-1
   411529a..fce19ec  main       -> origin/main
Updating 411529a..fce19ec
Fast-forward
 Pocket_Agent.ipynb                    | 1531 ++++++++++++++++++++++++++-------
 README.md                             |    6 +-
 __pycache__/inference.cpython-310.pyc |  Bin 0 -> 3701 bytes
 demo.py                               |    2 +-
 quantize/quantize.py                  |    4 +-
 train/finetune.py                     |    2 +-
 6 files changed, 1213 insertions(+), 332 deletions(-)
 create mode 100644 __pycache__/inference.cpython-310.pyc


In [ ]:
!pip uninstall -y torchvision

Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128


In [5]:
# Step 4: Fine-tune the Qwen model via QLoRA
!make train

python train/finetune.py
INFO:__main__:Loading dataset from /content/pocket-agent-cloud/train/../data/train.jsonl
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/json/json.py "HTTP/1.1 200 OK"
Generating train split: 1500 examples [00:00, 157050.82 examples/s]
INFO:__main__:Initializing Tokenizer...
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/config.json "HTTP/1.1 200 OK"
config.json: 100% 659/659 [00:00<00:00, 3.99MB/s]
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/tokenizer_config.

In [6]:
!make quantize

python quantize/quantize.py
INFO:__main__:Loading base model to merge adapter...
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"
Loading weights: 100% 338/338 [00:00<00:00, 855.42it/s, Materializing param=model.norm.weight]                               
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HT

In [7]:
# Step 6: Run Evaluation against test set
!make eval

python eval/evaluate.py
Test file not found at /content/pocket-agent-cloud/eval/../starter/public_test.jsonl. Running dummy evaluation to prove GGUF inference works!
INFO:inference:Loading GGUF model from /content/pocket-agent-cloud/artifacts/model.Q4_K_M.gguf
llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
Prompt: What is the weather in Tokyo?
Expected: {"tool": "weather", "args": {"location": "Tokyo", "unit": "celsius"}}
Predicted: <tool_call>{"tool": "weather", "args": {"location": "Tokyo", "units": "metric"}}</tool_call>
Latency: 12068.86 ms
--------------------------------------------------
Prompt: Convert 50 mph to kmh
Expected: {"tool": "convert", "args": {"value": 50.0, "from_unit": "mph", "to_unit": "kmh"}}
Predicted: <tool_call>{"tool": "convert", "args": {"value": 50, "from_unit": "mph", "to_unit": "kmh"}}</tool_call>
Latency: 6392.76 ms
--------------------------------------------------
Prompt: Turn on the smart h

### Step 7: Run Streamlit Demo
To view the Streamlit UI, we will use localtunnel to expose the port from the Colab kernel to your browser.

In [8]:
import urllib
print("Password/Endpoint IP for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

!streamlit run demo.py & npx localtunnel --port 8501

Password/Endpoint IP for localtunnel is: 34.178.98.84
⠙⠹

⠸⠼⠴⠦⠧Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.178.98.84:8501

  Stopping...
^C
